# 5. Ajuste de hiperparámetros y validación externa
El **tuneo** compara configuraciones de un algoritmo; no equivale a evaluar su generalización.
En cada entrenamiento externo se ejecuta `GridSearchCV` por familia sobre folds internos.
Se seleccionan hiperparámetros y familia por menor MAE interno, se reajusta sobre ese entrenamiento
y solo entonces se predice el fold externo. No se usa el error externo para elegir la familia.

Rejillas iniciales revisables: regresiones regularizadas, árbol, bosque y proceso gaussiano.
La red neuronal está preparada pero deshabilitada hasta justificar cantidad y representación.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src' / 'ingesta.py').is_file()
             and (p / 'config' / 'data_schema.yml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Abra Jupyter desde el repositorio o notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.experimento import read_config, inspect_plan, run_experiment
config, config_hash = read_config(ROOT / 'config' / 'model_config.yml')
inspect_plan(config)


## Ejecución explícita, desactivada en este avance
Esta celda sí entrenaría si se activa. Requiere datos curados reales, protocolo revisado y
`execution.allow_training: true`. Mantener desactivada mientras se prepara el avance.
El reajuste final opcional se realiza después de la evaluación; no crea una prueba independiente nueva.


In [ ]:
RUN_EXPERIMENT = False
FIT_FINAL_MODEL = False
if RUN_EXPERIMENT:
    run_dir = run_experiment(config, config_hash, fit_final=FIT_FINAL_MODEL)
    print('Artefactos locales:', run_dir)
else:
    print('No se entrenó ni evaluó ningún modelo.')


## Artefactos y control de decisiones
Se conservarán configuración, hashes, entorno, folds internos/externos, búsquedas, advertencias,
pipelines ajustados, baseline, predicciones fuera de muestra y métricas por fold.
`selected` evalúa el procedimiento completo de selección interna. Elegir una familia después de
ver la tabla de errores externos puede introducir optimismo: esa elección requiere nueva evaluación.

No tratar la desviación entre folds como un intervalo de confianza de observaciones independientes.
Continuar con `06_Evaluacion_e_interpretabilidad.ipynb` solo cuando exista una ejecución real completa.
